In [42]:
import math

from pyod.models import lof
from scipy.io import arff
from operator import index

import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import scipy.stats
from scipy.stats import expon, skew, norm,gamma, anderson,goodness_of_fit, monte_carlo_test, probplot, skewnorm
from scipy import integrate
from sklearn.metrics import auc
import seaborn as sns
import math
from pyod.models.cof import COF
from pyod.models.abod import ABOD
from statsmodels.sandbox.distributions.gof_new import kstest

plt.rcParams['figure.figsize'] = [15, 7]
import warnings
from scipy import stats
from pyod.models import abod, knn, lof, cof, kde, sos, sod
warnings.filterwarnings('ignore')

In [81]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
from scipy.stats import norm, gaussian_kde
from itertools import combinations

def simplified_lof(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, _ = nbrs.kneighbors(X)
    return distances[:, 1:].mean(axis=1)

# LDOF
def ldof(X, k):
    n = X.shape[0]
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    distances, indices = distances[:, 1:], indices[:, 1:]

    scores = np.zeros(n)
    for i in range(n):
        xi = X[i]
        neighbors = X[indices[i]]
        d_avg = np.mean(np.linalg.norm(neighbors - xi, axis=1, ord=1))
        d_pairwise = np.mean([np.linalg.norm(neighbors[a] - neighbors[b], ord=1) for a, b in combinations(range(k), 2)])
        scores[i] = d_avg / d_pairwise if d_pairwise != 0 else 0
    return scores

# ODIN
def odin(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, _ = nbrs.kneighbors(X)
    return distances[:, -1]

# KDEOS
def kdeos(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    _, indices = nbrs.kneighbors(X)
    scores = []
    for i in range(len(X)):
        local_data = X[indices[i, 1:]]
        if local_data.shape[0] < X.shape[1]:
            scores.append(0)  # Skip ill-posed KDE estimation
            continue
        try:
            kde = gaussian_kde(local_data.T)
            score = kde(X[i].reshape(-1, 1))
            scores.append(1 / (score[0] + 1e-10))
        except np.linalg.LinAlgError:
            scores.append(0)
    return np.array(scores)

# LDF
def ldf(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, _ = nbrs.kneighbors(X)
    density = 1 / (np.mean(distances[:, 1:], axis=1) + 1e-8)
    return 1 / density

# LoOP
def loop(X, k, lambda_val=3.0):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    distances, indices = distances[:, 1:], indices[:, 1:]

    pdists = np.zeros(X.shape[0])
    for i in range(X.shape[0]):
        xi = X[i]
        neighbors = X[indices[i]]
        sigma_sq = np.mean(np.square(np.linalg.norm(neighbors - xi, axis=1, ord=1)))
        pdists[i] = np.sqrt(sigma_sq)

    loop_scores = np.zeros(X.shape[0])
    for i in range(X.shape[0]):
        neighbor_pdists = pdists[indices[i]]
        mean_sigma = np.mean(neighbor_pdists)
        plof = (pdists[i] / (mean_sigma + 1e-8)) - 1
        loop_scores[i] = norm.cdf(plof * lambda_val)
    return loop_scores

# INFLO
def inflo(X, k):
    n = X.shape[0]
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    indices = indices[:, 1:]

    reverse_knn = [[] for _ in range(n)]
    for i in range(n):
        for j in indices[i]:
            reverse_knn[j].append(i)

    densities = np.zeros(n)
    for i in range(n):
        neighbors = X[indices[i]]
        densities[i] = 1 / (np.mean(np.linalg.norm(neighbors - X[i], axis=1, ord=1)) + 1e-8)

    scores = np.zeros(n)
    for i in range(n):
        inf_neighbors = list(set(indices[i]) | set(reverse_knn[i]))
        inf_density = np.mean(densities[inf_neighbors]) if inf_neighbors else 1e-8
        scores[i] = inf_density / (densities[i] + 1e-8)
    return scores

def abod2(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    n_samples = X.shape[0]

    scores = np.zeros(n_samples)

    for i in range(n_samples):
        neighbors = X[indices[i, 1:]]  # exclude itself
        diff = neighbors - X[i]  # shape (k, d)

        # Normalize difference vectors
        norms = np.linalg.norm(diff, axis=1)
        diff_normalized = diff / (norms[:, np.newaxis] + 1e-10)

        # Calculate angles between all pairs of difference vectors
        # cosine of angle between vectors = dot product
        # We'll compute dot products between all pairs (k choose 2)

        # Compute all pairwise dot products
        dot_prods = diff_normalized @ diff_normalized.T  # (k, k)

        # We only consider pairs (j < l), off-diagonal elements
        # Flatten and remove diagonal
        mask = ~np.eye(k, dtype=bool)
        dot_pairs = dot_prods[mask]

        # ABOD score is variance of these dot products (angles)
        scores[i] = np.var(dot_pairs)

    # Return inverse to match outlierness (low score = outlier)
    return 1 / (scores + 1e-8)

def cof2(X, k):
    nbrs = NearestNeighbors(n_neighbors=k+1, p=1).fit(X)
    distances, indices = nbrs.kneighbors(X)
    n_samples = X.shape[0]

    # For each point:
    # 1. get its neighbors (excluding itself)
    # 2. calculate the chaining distance by ordering neighbors by distance
    # 3. calculate average chaining distance and compare to neighbors' average chaining distance

    def chaining_distance(point_idx):
        neighbors_idx = indices[point_idx, 1:]
        neighbor_points = X[neighbors_idx]

        # Sort neighbors by distance from point_idx
        dists = distances[point_idx, 1:]
        order = np.argsort(dists)
        sorted_neighbors = neighbor_points[order]
        sorted_indices = neighbors_idx[order]

        # Chaining distance: sum of distances between successive neighbors plus distance from point to first neighbor
        total_dist = dists[order[0]]  # distance from point to first neighbor

        # distances between successive neighbors in the sorted order
        for i in range(len(sorted_neighbors) - 1):
            total_dist += np.linalg.norm(sorted_neighbors[i] - sorted_neighbors[i+1])

        avg_chaining_dist = total_dist / k
        return avg_chaining_dist

    chaining_dists = np.array([chaining_distance(i) for i in range(n_samples)])

    # Now calculate COF as ratio of chaining distance of point i to average chaining distances of its neighbors
    cof_scores = np.zeros(n_samples)
    for i in range(n_samples):
        neighbors_idx = indices[i, 1:]
        cof_scores[i] = chaining_dists[i] / (np.mean(chaining_dists[neighbors_idx]) + 1e-8)

    return cof_scores


# Evaluate AUC
def evaluate(y,scores):
    auc = roc_auc_score(y, scores)
    return auc

functions = [
    simplified_lof,
    ldof,
    odin,
    kdeos,
    ldf,
    loop,
    inflo,
    abod2, cof2
]

In [82]:
class ParametricMethodStateOfArt:
    def __init__(self,filename,p,logTrue=False):
        self.distance = []
        self.fileName = filename
        self.X = 0
        self.y = 0
        self.arr = []
        self.logTrue = logTrue
        self.p = p
        self.tots = []
        self.dataframe = pd.DataFrame()
        self.endValues = []

    def generateOutput(self,function):
        self._readArff()
        for a in range(1,70):
            scores = function(self.X,a)
            scores = np.nan_to_num(scores, nan=0.0)
            scores = list(scores)
            self.tots += [evaluate(self.y,scores)]
        return self._printResults(self.tots)


    def _readArff(self):
        arff_file = arff.loadarff(f'./{self.fileName}') # import the attribute-relation file format
        df4 = pd.DataFrame(arff_file[0])
        self.X = df4.drop(columns=['outlier','id']).values
        #get outlier values
        self.y = df4['outlier'].values
        le = LabelEncoder()
        #encoded the variables as 0=non-outlier, 1=outlier
        self.y = le.fit_transform(self.y)

    def _printResults(self,totalArr):
        newarr = np.nan_to_num(totalArr)
        newarr = list(newarr)
        #print(max(newarr),newarr.index(max(newarr))+2) #print the max values, the k value, and the array
        return max(newarr),newarr.index(max(newarr))+2

In [83]:
folder_structure_1d = [
    "semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff",
    "semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff",
    "semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff",
    "semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff",
    "semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff",
    "semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff",
    "semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff",
    "semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff",
    "semantic/Pima/Pima_withoutdupl_norm_35.arff",
    "semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff",
    "semantic/Stamps/Stamps_withoutdupl_norm_09.arff",
    "semantic/Wilt/Wilt_withoutdupl_norm_05.arff"
]

In [85]:
endvalues = []
df = pd.DataFrame(columns=['ROC AUC', 'k'])
for n in functions:
    endvalues = []
    for z in folder_structure_1d:
        parametricMethod = ParametricMethodStateOfArt(z,p=1,logTrue=True)
        endvalues += [parametricMethod.generateOutput(n)]
    print(endvalues)
    dftest = pd.DataFrame(np.array(endvalues),columns=['ROC AUC','k'])
    df = pd.concat([df,dftest])

df.to_csv("testForStateSemABOD.csv")

[(0.7133743926990428, 46), (0.6752944453286647, 70), (0.5, 2), (0.6010555555555556, 66), (0.6808266360505166, 28), (0.5483535805626598, 14), (0.5078146269771475, 70), (0.5819160997732427, 8), (0.5112835820895523, 23), (0.5470855071207243, 70), (0.7647979956154087, 69), (0.8503301678388718, 29)]
[(0.6261632777072632, 55), (0.7551925831609105, 41), (0.5692229363723488, 70), (0.5647222222222222, 70), (0.730195177956372, 51), (0.6849050443489144, 32), (0.699147904093032, 70), (0.7094671201814059, 70), (0.6615373134328357, 70), (0.48705175058993827, 3), (0.6350349723353168, 70), (0.5981581905676567, 2)]


In [28]:
# Assuming your list of tuples is stored in a variable called `data`
data = [
    (0.6774003117785861, 3), (0.7580773518536384, 51), (0.5377763135963998, 70), (0.6697222222222221, 70), (0.7588978185993112, 51), (0.7421173545736519, 18),
    (0.847513859256891, 70), (0.7210884353741497, 6), (0.7305373134328358, 70), (0.6402688931024345, 70), (0.9104290635756738, 70), (0.566802110628195, 3),
    (0.7891573176107234, 28), (0.7518144604879122, 6), (0.5617017896597024, 50), (0.5432222222222222, 5), (0.72904723306544, 69), (0.64681935078044, 41),
    (0.8021086079756498, 70), (0.5297619047619047, 23), (0.585029507462687, 65), (0.5, 2), (0.7069631485541288, 69), (0.698365398824468, 16),
    (0.6766873097300628, 2), (0.7606040180282554, 44), (0.557608787828659, 70), (0.6998888888888889, 69), (0.7898966794938654, 26), (0.722096754522501, 14),
    (0.8728066159906153, 70), (0.7373866213151927, 6), (0.7360373134328357, 68), (0.6504956282371211, 40), (0.9190983511332686, 64), (0.561776173251555, 2),
    (0.5, 2), (0.5, 2), (0.503218841201717, 36), (0.65425, 53), (0.70750810792192, 36), (0.5, 2), (0.649880738697281, 70), (0.7694160997732427, 57),
    (0.667865676146792, 70), (0.5, 2), (0.785102050428542, 70), (0.709467654692494, 62),
    (0.6774003117785861, 3), (0.7580773518536384, 51), (0.5377763135963998, 70), (0.6697222222222221, 70), (0.7588978185993112, 51), (0.7421173545736519, 18),
    (0.847513859256891, 70), (0.7210884353741497, 6), (0.7305373134328358, 70), (0.6402688931024345, 70), (0.9104290635756738, 70), (0.566802110628195, 3),
    (0.7209470914343199, 38), (0.7576049973515127, 70), (0.5683596191507979, 21), (0.5555, 70), (0.7416762342135477, 65), (0.6527656835675791, 70), (0.7742656595712214, 70), (0.5756082721088435, 19), (0.6153419521337343, 69), (0.47210689432375455, 3), (0.7732228365871246, 70), (0.6835272236087888, 14),
    (0.7131290713087033, 31), (0.752984263159255, 70), (0.5797845743572649, 69), (0.5631666666666668, 68), (0.746286567614618, 64), (0.680269178864885, 70),
    (0.75350314319548, 70), (0.5250850340136055, 11), (0.6162089552238805, 70), (0.5069373110501278, 2), (0.7368201273619375, 70), (0.7020557233925322, 6)
]


# Convert to CSV string format
csv_string = "Score,k\n" + "\n".join([f"{score},{k}" for score, k in data])

# Save to a CSV file (optional)
with open("outlier_scores.csv", "w") as f:
    f.write(csv_string)

# Or just print if you want to copy-paste to Google Sheets manually
print(csv_string)


Score,k
0.6774003117785861,3
0.7580773518536384,51
0.5377763135963998,70
0.6697222222222221,70
0.7588978185993112,51
0.7421173545736519,18
0.847513859256891,70
0.7210884353741497,6
0.7305373134328358,70
0.6402688931024345,70
0.9104290635756738,70
0.566802110628195,3
0.7891573176107234,28
0.7518144604879122,6
0.5617017896597024,50
0.5432222222222222,5
0.72904723306544,69
0.64681935078044,41
0.8021086079756498,70
0.5297619047619047,23
0.585029507462687,65
0.5,2
0.7069631485541288,69
0.698365398824468,16
0.6766873097300627,2
0.7606040180282554,44
0.557608787828659,70
0.6998888888888889,69
0.7898966794938654,26
0.722096754522501,14
0.8728066159906153,70
0.7373866213151927,6
0.7360373134328357,68
0.6504956282371211,40
0.9190983511332687,64
0.561776173251555,2
0.5,2
0.5,2
0.503218841201717,36
0.65425,53
0.70750810792192,36
0.5,2
0.649880738697281,70
0.7694160997732427,57
0.667865676146792,70
0.5,2
0.785102050428542,70
0.709467654692494,62
0.6774003117785861,3
0.7580773518536384,51
0.53777631

In [86]:
literature_dataset_paths = [
    "literature/ALOI/ALOI_withoutdupl_norm.arff",
    "literature/Glass/Glass_withoutdupl_norm.arff",
    "literature/Ionosphere/Ionosphere_withoutdupl_norm.arff",
    "literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff",
    "literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff",
    "literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff",
    "literature/Shuttle/Shuttle_withoutdupl_norm_v10.arff",
    "literature/Waveform/Waveform_withoutdupl_norm_v10.arff",
    "literature/WBC/WBC_withoutdupl_norm_v10.arff",
    "literature/WDBC/WDBC_withoutdupl_norm_v10.arff",
    "literature/WPBC/WPBC_withoutdupl_norm.arff"
]



In [87]:
endvalues = []
df = pd.DataFrame(columns=['ROC AUC', 'K'])
for n in functions:
    endvalues = []
    for z in literature_dataset_paths:
        parametricMethod = ParametricMethodStateOfArt(z,p=1,logTrue=True)
        endvalues += [parametricMethod.generateOutput(n)]
    print(endvalues)
    dftest = pd.DataFrame(np.array(endvalues),columns=['ROC AUC','k'])
    df = pd.concat([df,dftest])

df.to_csv("testForStateLitABOD.csv")

[(0.7665924574895937, 14), (0.5, 2), (0.9207407407407407, 69), (0.5896926721349112, 70), (0.9917840375586855, 60), (0.5, 2), (0.5, 2), (0.5230720909362849, 5), (0.7629107981220657, 13), (0.5, 2), (0.545159926729604, 4)]
[(0.76844479189599, 30), (0.8986449864498645, 62), (0.8801763668430335, 13), (0.6057093064512763, 69), (0.9647887323943662, 14), (0.9829356214459789, 69), (0.6302307692307693, 64), (0.7625037391564462, 58), (0.9896713615023474, 58), (0.9770308123249299, 64), (0.5064111596449203, 47)]


In [37]:
import pandas as pd
from io import StringIO
import csv

data_str = """
ROC AUC	k
0.677400312	3
0.758077352	51
0.537776314	70
0.669722222	70
0.758897819	51
0.742117355	18
0.847513859	70
0.721088435	6
0.730537313	70
0.640268893	70
0.910429064	70
0.566802111	3
0.789157318	28
0.75181446	6
0.56170179	50
0.543222222	5
0.729047233	69
0.646819351	41
0.802108608	70
0.529761905	23
0.585029507	65
0.5	2
0.706963149	69
0.698365399	16
0.67668731	2
0.760604018	44
0.557608788	70
0.699888889	69
0.789896679	26
0.722096755	14
0.872806616	70
0.737386621	6
0.736037313	68
0.650495628	40
0.919098351	64
0.561776173	2
0.5	2
0.5	2
0.503218841	36
0.65425	53
0.707508108	36
0.5	2
0.649880739	70
0.7694161	57
0.667865676	70
0.5	2
0.78510205	70
0.709467655	62
0.677400312	3
0.758077352	51
0.537776314	70
0.669722222	70
0.758897819	51
0.742117355	18
0.847513859	70
0.721088435	6
0.730537313	70
0.640268893	70
0.910429064	70
0.566802111	3
0.720947091	38
0.757604997	70
0.568359619	21
0.5555	70
0.741676234	65
0.652765684	70
0.77426566	70
0.575608272	19
0.615341952	69
0.472106894	3
0.773222837	70
0.683527224	14
0.713129071	31
0.752984263	70
0.579784574	69
0.563166667	68
0.746286568	64
0.680269179	70
0.753503143	70
0.525085034	11
0.616208955	70
0.506937311	2
0.736820127	70
0.702055723	6
"""

df = pd.read_csv(StringIO(data_str), sep="\t")
df['ROC AUC'] = df['ROC AUC'].astype(str)
flattened = df.to_numpy().flatten()

num_pairs_per_row = 12
num_values_per_row = num_pairs_per_row * 2
rows = [flattened[i:i + num_values_per_row] for i in range(0, len(flattened), num_values_per_row)]

# Example output of the first row, to paste into an Excel cell or CSV
print(rows[0])

dft = pd.DataFrame(rows)
dft.to_csv("endr.csv")

['0.677400312' 3 '0.758077352' 51 '0.537776314' 70 '0.669722222' 70
 '0.758897819' 51 '0.742117355' 18 '0.847513859' 70 '0.721088435' 6
 '0.730537313' 70 '0.640268893' 70 '0.910429064' 70 '0.566802111' 3]
